## Reflection Pattern

One of the agentic design patterns covered in Andrew Ng's course on DeepLearning.AI.

The core idea: instead of accepting the LLM's first response, you have it critique and improve its own output, then repeat that loop a few times. The result is usually noticeably better than a single-shot generation — especially for tasks like writing, code generation, or anything where quality matters more than speed.

There are two main flavors:
- **Same-model reflection** — the LLM generates, then critiques its own output
- **External feedback** — instead of self-critique, you use an actual validator (test runner, linter, SQL executor) as the feedback source

A few things to keep in mind when writing reflection prompts:
- Be specific about what to critique — vague prompts produce vague feedback
- Give the critic a clear role that's distinct from the generator
- Always cap the number of iterations (2–3 is usually enough)

**Tradeoff:** better output quality at the cost of more API calls. Not worth it for simple tasks, very worth it for anything high-stakes.

The loop looks like this:

```
Generate → Critique → Refine → [repeat] → Final Output
```

You generate a first draft, critique it against specific criteria, revise based on the feedback, and repeat until either the quality is good enough or you've hit your iteration limit.

### Same model vs. separate critic

With same-model reflection, the LLM plays both roles using different system prompts. It's simpler and cheaper, but the model can have consistent blind spots — it may miss the same types of errors every time.

Using a separate model (or a strongly different system prompt) for the critic tends to produce more objective feedback. You can also run a cheaper model as the critic if it's good at evaluation but doesn't need to generate long text.

### External feedback

For tasks where you can validate output objectively, skip the LLM critic entirely. If your generated code throws a `TypeError`, that's more useful feedback than anything an LLM would say. Same idea with SQL — run the query and use the actual error or execution time as the reflection input.

```
LLM → [Generate] → Code
Test runner → [Execute] → Error messages  
LLM → [Fix] → Corrected code
```

This is the most reliable form of reflection because the feedback is objective.

### Use Cases

Reflection works best when you can define what "better" looks like and quality matters more than speed.

**Writing** is the most obvious fit — essays, technical docs, customer-facing copy. Structure, clarity, and argument strength are easy to critique and improve iteratively.

**Code generation** is where external feedback really shines. Rather than asking the LLM to guess whether its code is correct, you just run it. Failed tests or linter errors become the reflection input automatically.

**SQL** follows the same pattern — generate a query, execute it, use errors or slow query times as feedback.

**Reasoning tasks** like mathematical proofs or multi-step analysis benefit from a second pass that checks logical consistency and catches gaps in the argument.

It's less useful for simple Q&A, quick lookups, or anything where the first draft is good enough. The extra API calls aren't worth it there.

### Benefits and Limitations

The main benefit is output quality. Multiple iterations catch errors, tighten arguments, and generally produce more polished results — especially for complex tasks where the model's first attempt tends to miss things. It also helps with hallucinations: giving the model a chance to review its own claims means it often catches and corrects inaccurate information before you see it.

The downside is cost and latency. A 3-iteration reflection loop might consume 4–6x more tokens than a single-shot generation. For high-volume or real-time applications this adds up fast. Quality improvements also plateau after 2–3 iterations, so there's a point of diminishing returns.

One risk that's easy to overlook: over-refinement. The model can make unnecessary changes, introduce new errors, or lose the original voice through too many passes. Setting a max iteration limit and a clear stopping condition (see the working example below) keeps this in check.

### Writing Good Reflection Prompts

The most important thing is specificity. "Review this essay" produces generic feedback. "Review this essay for logical consistency in the argument and clarity of transitions between paragraphs" produces something actionable.

A few things that work well in practice:

**Give the critic an explicit role.** The system prompt for the critic should read differently from the generator's — something like "You are reviewing this as an editor" rather than reusing the writer framing. This helps the model shift into evaluation mode.

**Ask for both what works and what doesn't.** Feedback that only identifies problems can lead to the generator over-correcting and losing what was already good. Knowing what to keep is as useful as knowing what to fix.

**Be specific about what dimensions to evaluate.** For writing: structure, argument strength, clarity, tone. For code: correctness, edge cases, readability. Giving the critic a concrete rubric produces more consistent feedback than open-ended prompts.

**Use a structured stopping signal.** Don't check if the feedback contains a word like "excellent" — that's fragile. Instead, instruct the critic to output `[CONTINUE]` or `[STOP]` explicitly at the end of its response. The working example below uses this approach.

**Tell the generator to preserve intent.** In the refinement prompt, explicitly say to maintain the core thesis and main arguments. Without this, the model tends to drift significantly over multiple iterations.

### Evaluating Whether Reflection Is Actually Helping

The simplest check: compare the final output against the first draft. Is it actually better, and by how much?

For tasks with objective ground truth (code, SQL, math), just measure pass rates or execution success. Run your tests against the initial draft and the final output — if reflection doesn't improve the pass rate, the extra cost isn't justified.

For subjective tasks like writing, an LLM-as-judge approach works reasonably well: give a separate model both versions without labeling which came first, and ask which is better and why. It won't be perfect, but it's faster and cheaper than human evaluation for iteration purposes.

If you're trying to optimize the system rather than just validate it, a few metrics worth tracking:
- Which iteration produces the biggest quality jump (usually iteration 1 → 2)
- How often the `[STOP]` signal fires vs. hitting max iterations
- Token cost per quality point gained — this varies a lot by task type

### Implementation Example: Essay Writing with Reflection

This example demonstrates the complete reflection workflow for essay writing.

In [ ]:
from typing import Dict, List, Tuple

def generate_draft(topic: str, model: str = "gpt-4o") -> str:
    """
    Step 1: Generate initial draft essay.

    Args:
        topic: The essay topic
        model: LLM model to use

    Returns:
        Initial draft essay
    """
    prompt = f"""You are an expert essay writer. Write a comprehensive, well-structured essay on the following topic.

Topic: {topic}

Instructions:
- Write a complete essay with introduction, body paragraphs, and conclusion
- Support your arguments with relevant examples and evidence
- Maintain clear logical flow and transitions
- Aim for depth and substance

Write the complete draft essay now:"""

    # In production: response = llm_client.generate(prompt=prompt, model=model)
    # For demo purposes, we'll show the prompt
    print(f"[GENERATING DRAFT] Topic: {topic}\n")
    return "[DRAFT ESSAY WOULD BE GENERATED HERE]"


def reflect_on_draft(draft: str, model: str = "o4-mini") -> str:
    """
    Step 2: Critically evaluate the draft and provide feedback.

    Args:
        draft: The essay draft to critique
        model: LLM model to use (can be different from generator)

    Returns:
        Constructive feedback in paragraph form
    """
    prompt = f"""You are an expert essay critic and writing coach. Carefully review this draft and provide critical but constructive feedback.

Essay Draft:
{draft}

Analyze the draft and provide feedback addressing:
1. Structure and Organization - logical flow, transitions, essay structure
2. Clarity and Coherence - readability, confusing passages
3. Strength of Argument - thesis support, reasoning quality, evidence
4. Writing Style - tone, word choice, sentence variety
5. Depth and Substance - comprehensiveness, missed aspects
6. Specific Issues - redundancy, weak examples, unsupported claims

Provide 2-3 paragraphs of specific, actionable feedback. Be honest about weaknesses while offering constructive suggestions."""

    # In production: response = llm_client.generate(prompt=prompt, model=model)
    print(f"\n[REFLECTING ON DRAFT]\n")
    return "[CRITICAL FEEDBACK WOULD BE GENERATED HERE]"


def refine_draft(original_draft: str, reflection: str, model: str = "gpt-4o") -> str:
    """
    Step 3: Improve the draft based on reflection feedback.

    Args:
        original_draft: Initial essay version
        reflection: Critique feedback
        model: LLM model to use

    Returns:
        Refined essay
    """
    prompt = f"""You are an expert essay writer. Revise this draft based on the constructive feedback provided.

Original Draft:
{original_draft}

Feedback:
{reflection}

Instructions:
- Address all issues mentioned in the feedback
- Improve structure, clarity, arguments, and style
- Maintain the core thesis and main ideas
- Output ONLY the revised essay (no explanations)

Revised Essay:"""

    # In production: response = llm_client.generate(prompt=prompt, model=model)
    print(f"\n[REFINING DRAFT]\n")
    return "[REFINED ESSAY WOULD BE GENERATED HERE]"


print("Reflection workflow functions defined")

In [ ]:
def reflection_loop(
    topic: str,
    max_iterations: int = 3,
    generator_model: str = "gpt-4o",
    critic_model: str = "o4-mini"
) -> Dict[str, any]:
    """
    Complete reflection workflow with multiple iterations.

    Args:
        topic: Essay topic
        max_iterations: Maximum refinement cycles
        generator_model: Model for generation and refinement
        critic_model: Model for critique (can be same or different)

    Returns:
        Dictionary containing all versions and feedback
    """
    print("=" * 70)
    print(f"REFLECTION LOOP: {topic}")
    print("=" * 70)

    # Step 1: Generate initial draft
    current_draft = generate_draft(topic, generator_model)

    history = {
        "topic": topic,
        "iterations": [],
        "final_output": current_draft
    }

    # Iterative refinement loop
    for iteration in range(max_iterations):
        print(f"\n{'=' * 70}")
        print(f"Iteration {iteration + 1}/{max_iterations}")
        print('=' * 70)

        # Step 2: Reflect on current draft
        feedback = reflect_on_draft(current_draft, critic_model)

        # Check if reflection suggests no major issues (stopping condition)
        # In production, you'd parse the feedback to determine if iteration should stop
        if "excellent" in feedback.lower() and iteration > 0:
            print("\nQuality threshold met. Stopping iterations.")
            break

        # Step 3: Refine based on feedback
        refined_draft = refine_draft(current_draft, feedback, generator_model)

        # Store iteration history
        history["iterations"].append({
            "iteration": iteration + 1,
            "draft": current_draft,
            "feedback": feedback,
            "refined": refined_draft
        })

        # Update current draft for next iteration
        current_draft = refined_draft
        history["final_output"] = current_draft

        print(f"\nIteration {iteration + 1} complete")

    print("\n" + "=" * 70)
    print(f"REFLECTION COMPLETE: {len(history['iterations'])} iterations")
    print("=" * 70)

    return history


# Example usage
result = reflection_loop(
    topic="The impact of artificial intelligence on modern education",
    max_iterations=2,
    generator_model="gpt-4o",
    critic_model="o4-mini"
)

print(f"\nSUMMARY:")
print(f"   Topic: {result['topic']}")
print(f"   Iterations: {len(result['iterations'])}")
print(f"   Final output ready: {'final_output' in result}")

### Variations Worth Knowing

**Multi-aspect reflection** — run separate critique passes for different dimensions and combine the feedback. For example, one pass for structure, another for factual accuracy, another for style. More thorough but proportionally more expensive.

```python
def multi_aspect_reflection(draft):
    structure_feedback = reflect_on_structure(draft)
    accuracy_feedback = reflect_on_accuracy(draft)
    style_feedback = reflect_on_style(draft)
    return combine_feedback([structure_feedback, accuracy_feedback, style_feedback])
```

**Confidence-based reflection** — only trigger reflection when the model signals low confidence in its output. Saves tokens on cases where the first draft is already good.

```python
def conditional_reflection(draft, confidence_threshold=0.8):
    confidence = estimate_confidence(draft)
    if confidence < confidence_threshold:
        return reflect_and_refine(draft)
    return draft
```

**External validator** — for code, skip the LLM critic and use actual test results as feedback. This is the most reliable approach for verifiable outputs.

```python
def code_reflection_with_tests(problem):
    draft_code = generate_code(problem)
    test_results = run_unit_tests(draft_code)
    if test_results.failures:
        feedback = f"Tests failed: {test_results.error_messages}"
        return refine_code(draft_code, feedback)
    return draft_code
```

**Comparative reflection** — generate two drafts with different approaches, then synthesize the best elements from each. Useful when you're not sure which direction to take upfront.

```python
def comparative_reflection(topic):
    draft_a = generate_draft(topic, style="formal")
    draft_b = generate_draft(topic, style="casual")
    return synthesize_best_version([draft_a, draft_b])
```

**Hierarchical reflection** — reflect at multiple levels of granularity: overall structure first, then paragraph-level, then sentence-level. Each pass focuses on finer detail. Expensive, but useful for long-form documents where broad structure and fine-grained style both matter.

```python
def hierarchical_reflection(essay):
    structure_feedback = reflect_on_overall_structure(essay)
    essay = refine_structure(essay, structure_feedback)
    for paragraph in essay.paragraphs:
        para_feedback = reflect_on_paragraph(paragraph)
        paragraph = refine_paragraph(paragraph, para_feedback)
    return essay
```

### Real-World Example: Code Generation with External Feedback

This example shows reflection with objective external validation.

In [ ]:
import subprocess
import tempfile
import os

def generate_python_code(task_description: str) -> str:
    """Generate Python code for a given task."""
    prompt = f"""Write Python code to accomplish the following task:

Task: {task_description}

Requirements:
- Write clean, well-documented code
- Include proper error handling
- Follow PEP 8 style guidelines
- Include a main function if applicable

Return only the Python code, no explanations."""

    # Simulated response
    code = '''def fibonacci(n):
    if n <= 0:
        return []
    elif n == 1:
        return [0]
    elif n == 2:
        return [0, 1]

    fib = [0, 1]
    for i in range(2, n):
        fib.append(fib[i-1] + fib[i-2])
    return fib

if __name__ == "__main__":
    result = fibonacci(10)
    print(result)'''

    return code


def execute_python_code(code: str) -> Dict[str, any]:
    """Execute Python code and capture results/errors."""
    try:
        with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
            f.write(code)
            temp_file = f.name

        result = subprocess.run(
            ['python3', temp_file],
            capture_output=True,
            text=True,
            timeout=5
        )

        os.unlink(temp_file)

        return {
            "success": result.returncode == 0,
            "stdout": result.stdout,
            "stderr": result.stderr,
            "return_code": result.returncode
        }
    except subprocess.TimeoutExpired:
        return {
            "success": False,
            "error": "Code execution timeout"
        }
    except Exception as e:
        return {
            "success": False,
            "error": str(e)
        }


def reflect_on_code_errors(code: str, execution_result: Dict) -> str:
    """Generate reflection based on execution errors."""
    if execution_result["success"]:
        return "Code executed successfully. No errors to address."

    error_message = execution_result.get("stderr", execution_result.get("error", "Unknown error"))

    reflection = f"""The code has execution issues:

Error: {error_message}

Analysis needed:
1. What is causing this error?
2. How can the code be fixed?
3. Are there edge cases not handled?
4. Does the logic correctly implement the requirements?

Provide a corrected version of the code."""

    return reflection


def code_reflection_workflow(task: str, max_iterations: int = 3) -> Dict:
    """Complete code generation workflow with external validation."""
    print("=" * 70)
    print("CODE GENERATION WITH REFLECTION")
    print(f"Task: {task}")
    print("=" * 70)

    current_code = generate_python_code(task)
    print(f"\nInitial code generated ({len(current_code)} characters)")

    for iteration in range(max_iterations):
        print(f"\n--- Iteration {iteration + 1} ---")

        # Execute code
        print("Executing code...")
        result = execute_python_code(current_code)

        if result["success"]:
            print("Code executed successfully!")
            if result["stdout"]:
                print(f"Output: {result['stdout'].strip()}")
            return {
                "success": True,
                "code": current_code,
                "iterations": iteration + 1,
                "output": result["stdout"]
            }

        # Reflect on errors
        print(f"Execution failed: {result.get('stderr', 'Error')[:100]}...")
        feedback = reflect_on_code_errors(current_code, result)

        # Refine code based on feedback
        print("Refining code based on errors...")
        # In production: current_code = refine_code(current_code, feedback)
        # For demo, we'll simulate fixing the code
        current_code = current_code.replace("fib[i-1]", "fib[-1]").replace("fib[i-2]", "fib[-2]")

    print(f"\nMax iterations ({max_iterations}) reached without success")
    return {
        "success": False,
        "code": current_code,
        "iterations": max_iterations
    }


# Example usage
result = code_reflection_workflow("Generate the first N Fibonacci numbers")

print(f"\nRESULT:")
print(f"   Success: {result['success']}")
print(f"   Iterations: {result['iterations']}")
if result['success'] and 'output' in result:
    print(f"   Output: {result['output']}")

### When to Use It

Use reflection when accuracy matters more than speed. Legal documents, code that needs to be correct, financial reports, customer-facing content — these are the right cases. The extra API calls are worth it when the cost of a bad output is high.

It's also a natural fit when you have external validation available. Code you can run, SQL you can execute, facts you can check against a source — these give you objective feedback that's more reliable than LLM self-critique.

Skip it for simple tasks, real-time interactions, or anything where the first draft is usually good enough. Also skip it when you can't define what "better" looks like — if you don't have clear quality criteria, the model won't either, and you'll just be burning tokens on vague iterations.

### How It Fits with Other Patterns

| Pattern | Purpose | Key Difference |
|---------|---------|----------------|
| **Reflection** | Improve output quality through self-critique | Iterative refinement of a single output |
| **Tool Use** | Extend capabilities with external systems | Access to external data/functionality |
| **Planning** | Break complex tasks into steps | Task decomposition, not quality improvement |
| **RAG** | Ground responses in external knowledge | Information retrieval, not self-improvement |
| **Multi-Agent** | Distribute work across specialized agents | Parallel processing, not sequential refinement |
| **Chain-of-Thought** | Show reasoning step-by-step | Transparency in a single pass, not iteration |

Reflection composes naturally with these other patterns. The most common combination is reflection + external tool feedback — generate code, run it, use the errors as the reflection input:

```python
query = generate_sql(question)
result = execute_query(query)
if result.has_errors:
    feedback = reflect_on_errors(query, result)
    query = refine_query(query, feedback)
```

Reflection + RAG is also useful: retrieve documents, generate an answer, then have the critic verify the answer against the retrieved content before returning it.

### Summary

Reflection is the simplest agentic pattern to implement and one of the most immediately useful. The core loop is three functions: generate, critique, refine. The main decisions when building it are:

- Same model or separate critic?
- How many iterations? (2–3 is usually enough)
- What stopping condition? (structured signal beats keyword matching)
- Is there external validation you can use instead of LLM self-critique?

The most common mistake is not being specific enough in the reflection prompt. Vague critique ("improve this essay") leads to vague revisions. Give the critic concrete dimensions to evaluate.

### Papers

- **Self-Refine** (Madaan et al., 2023) — the iterative refinement framework this pattern is based on
- **Reflexion** (Shinn et al., 2023) — reflection for autonomous agents with verbal reinforcement learning
- **Constitutional AI** (Anthropic, 2022) — self-critique applied to alignment
- **Self-Debugging** (Chen et al., 2023) — code generation with reflection

### Workflow Diagram

```
┌─────────────────────────────────────────────────────────────┐
│                     USER REQUEST                            │
│          "Write an essay on climate change"                 │
└─────────────────────┬───────────────────────────────────────┘
                      │
                      ▼
┌─────────────────────────────────────────────────────────────┐
│              STEP 1: GENERATION                             │
│                                                             │
│  LLM generates initial draft essay                          │
│  • Introduction, body, conclusion                           │
│  • Basic structure and arguments                            │
└─────────────────────┬───────────────────────────────────────┘
                      │
                      ▼
┌─────────────────────────────────────────────────────────────┐
│               DRAFT VERSION 1                               │
│  "Climate change is one of the most pressing issues..."     │
│  [Full essay draft - 500 words]                             │
└─────────────────────┬───────────────────────────────────────┘
                      │
                      ▼
┌─────────────────────────────────────────────────────────────┐
│              STEP 2: REFLECTION                             │
│                                                             │
│  LLM (or different LLM) critiques the draft:                │
│  • Structure: Weak transitions between paragraphs           │
│  • Arguments: Lacks supporting evidence                     │
│  • Clarity: Technical terms not explained                   │
│  • Style: Inconsistent tone                                 │
└─────────────────────┬───────────────────────────────────────┘
                      │
                      ▼
┌─────────────────────────────────────────────────────────────┐
│              CRITICAL FEEDBACK                              │
│  "The essay has a solid foundation but needs improvement    │
│   in three areas: 1) Add smoother transitions...            │
│   2) Include specific data and examples... 3) Define        │
│   technical terms for general audience...  [CONTINUE]"      │
└─────────────────────┬───────────────────────────────────────┘
                      │
                      ▼
┌─────────────────────────────────────────────────────────────┐
│              STEP 3: REFINEMENT                             │
│                                                             │
│  LLM improves draft based on feedback:                      │
│  • Adds transition sentences                                │
│  • Incorporates evidence and examples                       │
│  • Defines technical terms                                  │
│  • Harmonizes tone                                          │
└─────────────────────┬───────────────────────────────────────┘
                      │
                      ▼
┌─────────────────────────────────────────────────────────────┐
│               DRAFT VERSION 2                               │
│  [Improved essay with addressed issues]                     │
└─────────────────────┬───────────────────────────────────────┘
                      │
                      ▼
                  ┌───┴───┐
                  │Repeat?│  iterations < max_iterations
                  │       │  AND critic output [CONTINUE]
                  └───┬───┘
                      │
                      ▼
┌─────────────────────────────────────────────────────────────┐
│                 FINAL OUTPUT                                │
│  Refined essay — critic output [STOP] or max iters reached  │
└─────────────────────────────────────────────────────────────┘
```

---

### Working Implementation with Real LLM Calls

The preceding skeleton examples use placeholder strings. This section shows a complete, runnable implementation with three elements directly from Andrew Ng's course:

1. **Message history as conversation context** — the generator's full conversation (initial prompt → draft → critique → revision request) is passed as a message list on each API call, giving the model complete context of prior turns

2. **Structured `[CONTINUE]` / `[STOP]` signals** — the critic is instructed to end its response with one of two explicit tokens, making the loop termination condition deterministic instead of relying on fragile keyword matching like `"excellent" in feedback`

3. **Separate critic call** — the critic only sees the current draft (not the generation history), keeping the critique objective and focused

**Setup**: set `OPENAI_API_KEY` in your environment before running.

In [ ]:
import os
from openai import OpenAI

# Requires OPENAI_API_KEY set in your environment
# export OPENAI_API_KEY="sk-..."
client = OpenAI()

GENERATION_SYSTEM_PROMPT = """You are an expert essay writer. Write compelling, \
well-structured essays. When asked to revise, carefully address all feedback provided."""

REFLECTION_SYSTEM_PROMPT = """You are an expert essay critic and writing coach. \
Analyze the essay and provide specific, actionable feedback on structure, clarity, \
argument strength, and style.

End your critique with exactly one of:
- [CONTINUE] if the essay still has meaningful room for improvement
- [STOP] if the essay is well-written and no major improvements remain"""


def run_reflection_loop(topic: str, max_iterations: int = 3) -> str:
    """
    Reflection loop using full message history as conversation context.

    The generator sees its own previous drafts and each critique in a single
    conversation thread — this is the message-history pattern from the course.

    The critic outputs a structured [CONTINUE] / [STOP] signal so the loop
    terminates cleanly without brittle keyword guessing.
    """
    generation_messages = [
        {"role": "system", "content": GENERATION_SYSTEM_PROMPT},
        {"role": "user", "content": f"Write a comprehensive essay on: {topic}"},
    ]

    # Step 1: Generate initial draft
    print(f"[STEP 0] Generating initial draft on: {topic}")
    response = client.chat.completions.create(model="gpt-4o", messages=generation_messages)
    draft = response.choices[0].message.content
    generation_messages.append({"role": "assistant", "content": draft})
    print(f"         Draft: {len(draft)} characters\n")

    for i in range(max_iterations):
        # Step 2: Reflect — critic receives only the current draft
        print(f"[STEP {i + 1}a] Reflecting...")
        reflection_response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": REFLECTION_SYSTEM_PROMPT},
                {"role": "user", "content": draft},
            ],
        )
        critique = reflection_response.choices[0].message.content
        print(f"          Critique: {len(critique)} characters")

        # Structured stopping condition — no keyword guessing
        if "[STOP]" in critique:
            print(f"          [STOP] received — quality threshold met after {i + 1} reflection(s).\n")
            break

        # Step 3: Refine — append critique to generator history so it has full context
        print(f"[STEP {i + 1}b] Refining...")
        generation_messages.append({
            "role": "user",
            "content": f"Revise the essay based on this critique:\n\n{critique}",
        })
        refine_response = client.chat.completions.create(
            model="gpt-4o", messages=generation_messages
        )
        draft = refine_response.choices[0].message.content
        generation_messages.append({"role": "assistant", "content": draft})
        print(f"          Refined draft: {len(draft)} characters\n")

    return draft


# Run — requires OPENAI_API_KEY to be set
topic = "The impact of artificial intelligence on modern education"
final_essay = run_reflection_loop(topic, max_iterations=3)

print("=" * 70)
print("FINAL ESSAY (first 600 chars):")
print("=" * 70)
print(final_essay[:600] + "..." if len(final_essay) > 600 else final_essay)